# some of these functions are throughoughly tested and some are not, validate results before using in critical applications

In [ ]:
import numpy as np
import sympy
import scipy as sp

In [ ]:
def magnitude(vector): 
    return math.sqrt(sum(pow(element, 2) for element in vector))

def distance_plane_to_point(plane, point):
    return np.abs(plane[0]*point[0] + plane[1]*point[1] + plane[2]*point[2] + plane[3]) / np.sqrt(plane[0]**2 + plane[1]**2+  plane[2]**2) # formula for distance from a point to a plane

def analyze_critical_points(f, vars):
    """
    Finds and classifies critical points of f(x), f(x,y), or f(x,y,z).
    
    Parameters:
        f    : A symbolic function in SageMath.
        vars : A list of variables (e.g., [x], [x,y], or [x,y,z]).
    
    Returns:
        A dictionary of critical points with their classifications.
    """
    #from sympy import Matrix

    num_vars = len(vars)
    
    # Compute first-order partial derivatives (gradient)
    grad = [diff(f, var) for var in vars]
    
    # Solve for critical points
    crit_solutions = solve(grad, vars, solution_dict=True)
    
    if not crit_solutions:
        print("No critical points found.")
        return {}

    critical_points = []
    classifications = {}

    for sol in crit_solutions:
        try:
            point = tuple(sol[var] for var in vars)  # Ensure we get a full (x,y) or (x,y,z) point
        except KeyError:
            print(f"Skipping incomplete solution: {sol}")
            continue

        critical_points.append(point)

        if num_vars == 2:
            # Compute second-order partial derivatives
            f_xx = diff(grad[0], vars[0])
            f_yy = diff(grad[1], vars[1])
            f_xy = diff(grad[0], vars[1])

            # Compute Hessian determinant (Discriminant)
            D = f_xx * f_yy - f_xy**2

            # Evaluate at the critical point
            D_val = D.subs(sol)
            f_xx_val = f_xx.subs(sol)

            if D_val > 0:
                if f_xx_val > 0:
                    classifications[point] = "Local Minimum"
                elif f_xx_val < 0:
                    classifications[point] = "Local Maximum"
            elif D_val < 0:
                classifications[point] = "Saddle Point"
            else:
                classifications[point] = "Inconclusive (Hessian determinant = 0)"

    return classifications

def closest_point_on_function(f, point):
    from scipy.optimize import minimize
    """
    Find the point on the function f that is closest to the given point in space.
    
    Parameters:
    - f: The function (in 2D: f(x), in 3D: f(x, y)).
    - point: The point in space (in 2D: (x0, y0), in 3D: (x0, y0, z0)).
    - initial_guess: Initial guess for the optimization (in 2D: x_guess, in 3D: (x_guess, y_guess)).
    
    Returns:
    - The point on the function f that is closest to the given point.
    """
    if len(point) == 2:
        # 2D case: f is a function of one variable
        initial_guess = (0.0)
        x0, y0 = point
        def distance_squared(x):
            return (x[0] - x0)**2 + (f(x[0]) - y0)**2
        
        # Minimize the distance squared
        result = minimize(distance_squared, [initial_guess])
        x_min = result.x[0]
        return (x_min, f(x_min))
    
    elif len(point) == 3:
        initial_guess = (0.0,0.0)
        # 3D case: f is a function of two variables
        x0, y0, z0 = point
        def distance_squared(vars):
            x, y = vars
            return (x - x0)**2 + (y - y0)**2 + (f(x, y) - z0)**2
        
        # Minimize the distance squared
        result = minimize(distance_squared, initial_guess)
        x_min, y_min = result.x
        return (x_min, y_min, f(x_min, y_min))
    
    else:
        raise ValueError("The point must be in 2D or 3D space.")

def error_function(f, point):
    # Define the variables
    x, y = var('x y')
    
    # Extract the point (x0, y0)
    x0, y0 = point
    
    # Compute f(x0, y0)
    f0 = f.subs(x=x0, y=y0)
    
    # Compute partial derivatives fx and fy
    fx = f.diff(x)
    fy = f.diff(y)
    
    # Evaluate partial derivatives at (x0, y0)
    fx0 = fx.subs(x=x0, y=y0)
    fy0 = fy.subs(x=x0, y=y0)
    
    # Compute the linear approximation
    linear_approx = f0 + fx0 * (x - x0) + fy0 * (y - y0)
    
    # Compute the error function E(x, y)
    E = f - linear_approx
    
    return E

def linear_approximation(f, point):
    # Determine the number of variables in f
    variables = f.variables()
    n = len(variables)
    
    if n == 1:
        # Case for f(x): single-variable function
        x = variables[0]
        fx = f.diff(x)
        
        # Evaluate function and derivative at the point
        x0 = point[0]
        f0 = f.subs(x=x0)
        fx0 = fx.subs(x=x0)
        
        # Linear approximation: L(x) = f(x0) + f'(x0)*(x - x0)
        linear_approx = f0 + fx0 * (x - x0)
    
    elif n == 2:
        # Case for f(x, y): two-variable function
        x, y = variables
        fx = f.diff(x)
        fy = f.diff(y)
        
        # Evaluate function and partial derivatives at the point
        x0, y0 = point
        f0 = f.subs(x=x0, y=y0)
        fx0 = fx.subs(x=x0, y=y0)
        fy0 = fy.subs(x=x0, y=y0)
        
        # Linear approximation: L(x, y) = f(x0, y0) + fx0*(x - x0) + fy0*(y - y0)
        linear_approx = f0 + fx0 * (x - x0) + fy0 * (y - y0)
    
    elif n == 3:
        # Case for f(x, y, z): three-variable function
        x, y, z = variables
        fx = f.diff(x)
        fy = f.diff(y)
        fz = f.diff(z)
        
        # Evaluate function and partial derivatives at the point
        x0, y0, z0 = point
        f0 = f.subs(x=x0, y=y0, z=z0)
        fx0 = fx.subs(x=x0, y=y0, z=z0)
        fy0 = fy.subs(x=x0, y=y0, z=z0)
        fz0 = fz.subs(x=x0, y=y0, z=z0)
        
        # Linear approximation: L(x, y, z) = f(x0, y0, z0) + fx0*(x - x0) + fy0*(y - y0) + fz0*(z - z0)
        linear_approx = f0 + fx0 * (x - x0) + fy0 * (y - y0) + fz0 * (z - z0)
    
    else:
        raise ValueError("The function must have 1, 2, or 3 variables.")
    
    return linear_approx

def line_integral(f_or_F, C, t, a=None, b=None):
    """
    Compute the line integral of a scalar function or a vector field along a parameterized curve C.
    :param f_or_F: Scalar function f(x, y, z, ...) or vector field F (list of components as functions of x, y, z, ...)
    :param C: Parameterized curve (list of expressions for x(t), y(t), z(t), ...)
    :param t: Parameter of the curve
    :param a: Lower limit of integration
    :param b: Upper limit of integration
    :return: Line integral value
    Example (scalar function):
    f(x, y, z) = x^2 + y^2 + z
    C = [3*cos(t), 3*sin(t), t]
    var('t')
    line_integral(f, C, t, 0, pi)
    
    Example (vector field):
    F = [-z, 2*y, 4*x]
    C = [sin(t), cos(t), t]
    var('t')
    line_integral(F, C, t, 0, pi)
    """
    # Check if the input is a vector field (list of components)

    if isinstance(f_or_F, list):
        F = f_or_F
        dC = [diff(expr, t) for expr in C]
        # Only use as many variables as needed for the parameterization
        all_vars = [x, y, z]
        vars_list = all_vars[:len(C)]
        # Substitute only the variables present in C
        F_substituted = [component.subs(dict(zip(vars_list, C))) for component in F[:len(C)]]
        # Dot product only over the length of C
        integrand = sum(F_substituted[i] * dC[i] for i in range(len(C)))
    else:
        # Scalar function case
        f = f_or_F
        # Substitute the parameterized curve into the scalar function
        substituted_f = f(*C)

        # Compute the derivative of the parameterized curve
        path_integrand = sqrt(sum([diff(expr, t)^2 for expr in C]))

        # Compute the integrand
        integrand = substituted_f * path_integrand

    # Perform the integration
    if a is not None and b is not None:
        # Definite integral
        result = integrate(integrand, (t, a, b))
    else:
        # Indefinite integral
        result = integrate(integrand, t)

    return result

def gradient_vector_field(f, x, y, z):
    """
    Compute the gradient vector field of a scalar function f.
    :param f: Scalar function
    :param x: x-coordinate
    :param y: y-coordinate
    :param z: z-coordinate
    :return: Gradient vector field
    """
    # Compute the gradient vector field
    grad_f = [diff(f, var) for var in (x, y, z)]
    return grad_f


def flux(F, C, t, a, b):
    """
    Compute the flux of vector field F across curve C from t=a to t=b.
    F: vector field as a list, e.g. [x, y, 0]
    C: parameterized curve as a list, e.g. [4*cos(t), 4*sin(t), 0]
    t: parameter
    a, b: integration limits
    todo:
    rewite to use line_integral and other premade functions
    """
    # Compute the tangent vector dC/dt
    dC = [diff(expr, t) for expr in C]
    # Compute the magnitude of the tangent vector
    dC_mag = sqrt(sum(dc**2 for dc in dC))
    # Compute the unit tangent vector
    T_hat = [dc/dC_mag for dc in dC]
    # Compute the unit normal vector in the xy-plane: [-T_y, T_x, 0]
    n_hat = [T_hat[1], -T_hat[0], 0]
    # Substitute the parameterized curve into the vector field
    F_sub = [component.subs(x=C[0], y=C[1], z=C[2]) for component in F]
    # Compute dot product F·n
    integrand = sum(F_sub[i] * n_hat[i] for i in range(len(F)))
    # Integrate (F·n) * |dC/dt| dt over t
    return integrate(integrand * dC_mag, (t, a, b))

def is_conservative_2d(F):
    """
    Check if a 2D vector field F = [M(x, y), N(x, y)] is conservative.
    Returns True if conservative, False otherwise.
    """
    var('x y')
    M, N = F
    dM_dy = diff(M, y)
    dN_dx = diff(N, x)
    print("∂M/∂y =", dM_dy)
    print("∂N/∂x =", dN_dx)
    return bool(simplify(dM_dy - dN_dx) == 0)

def curl(F, vars):
    """
    Compute the curl of a vector field F (as a list) with respect to variables vars (as a list).
    - In 2D: returns scalar curl (∂N/∂x - ∂M/∂y)
    - In 3D: returns vector curl
    - In nD: returns list of all 2-form components (exterior derivative)
    """
    n = len(vars)
    if n == 2:
        # F = [M, N]
        return diff(F[1], vars[0]) - diff(F[0], vars[1])
    elif n == 3:
        # F = [M, N, P]
        return [
            diff(F[2], vars[1]) - diff(F[1], vars[2]),
            diff(F[0], vars[2]) - diff(F[2], vars[0]),
            diff(F[1], vars[0]) - diff(F[0], vars[1])
        ]
    else:
        # General exterior derivative (all 2-forms)
        return [diff(F[j], vars[i]) - diff(F[i], vars[j])
                for i in range(n) for j in range(i+1, n)]

def greens_theorem(F, vars, region):
    """
    Compute the line integral ∮_C F·dr using Green's Theorem over region.
    F: [M(x, y), N(x, y)]
    vars: [x, y]
    region: tuple of integration bounds, e.g. (x, 0, 1, y, 0, x - x^2)
    """
    assert len(vars) == 2, "Green's theorem is for 2D regions."
    curl_val = curl(F, vars)
    x, y = vars
    # region: (x, x0, x1, y, y0, y1)
    # Integrate with respect to y first, then x (outermost)
    return integrate(
        integrate(curl_val, (region[3], region[4], region[5])),
        (region[0], region[1], region[2])
    )


def unit_normal_vector(f, point):
    '''returns unit normal vector of sagemath function f, can be f(x,y) or f(x,y,x) number of variables in function must match number of 
     constants in point '''
    # Determine the number of variables in f
    variables = f.variables()
    n = len(variables)
    
    if n == 2:
        # Case for f(x, y): surface is z = f(x, y)
        x, y = variables
        fx = f.diff(x)
        fy = f.diff(y)
        
        # Evaluate partial derivatives at the point
        x0, y0 = point
        fx0 = fx.subs(x=x0, y=y0)
        fy0 = fy.subs(x=x0, y=y0)
        
        # Gradient vector is [fx, fy, -1]
        gradient_vector = vector([fx0, fy0, -1])
    
    elif n == 3:
        # Case for f(x, y, z): surface is f(x, y, z) = 0
        x, y, z = variables
        fx = f.diff(x)
        fy = f.diff(y)
        fz = f.diff(z)
        
        # Evaluate partial derivatives at the point
        x0, y0, z0 = point
        fx0 = fx.subs(x=x0, y=y0, z=z0)
        fy0 = fy.subs(x=x0, y=y0, z=z0)
        fz0 = fz.subs(x=x0, y=y0, z=z0)
        
        # Gradient vector is [fx, fy, fz]
        gradient_vector = vector([fx0, fy0, fz0])
    
    else:
        raise ValueError("The function must have 2 or 3 variables.")
    
    # Normalize the gradient vector to get the unit normal vector
    magnitude = gradient_vector.norm()
    unit_normal = gradient_vector / magnitude
    
    return np.array(unit_normal)

def normal_line_equations(f, point):
    '''returns tuple of normal line equations for sage symbolic expression f at list point, works for f(x, y) and point [_, _] or f(x, y, z) and point [_, _, _]'''
    # Determine the number of variables in f
    variables = f.variables()
    n = len(variables)
    
    if n == 2:
        # Case for f(x, y): surface is z = f(x, y)
        x, y = variables
        fx = f.diff(x)
        fy = f.diff(y)
        
        # Evaluate partial derivatives at the point
        x0, y0 = point
        z0 = f.subs(x=x0, y=y0)
        fx0 = fx.subs(x=x0, y=y0)
        fy0 = fy.subs(x=x0, y=y0)
        
        # Direction vector is [fx, fy, -1]
        direction_vector = vector([fx0, fy0, -1])
    
    elif n == 3:
        # Case for f(x, y, z): surface is f(x, y, z) = 0
        x, y, z = variables
        fx = f.diff(x)
        fy = f.diff(y)
        fz = f.diff(z)
        
        # Evaluate partial derivatives at the point
        x0, y0, z0 = point
        fx0 = fx.subs(x=x0, y=y0, z=z0)
        fy0 = fy.subs(x=x0, y=y0, z=z0)
        fz0 = fz.subs(x=x0, y=y0, z=z0)
        
        # Direction vector is [fx, fy, fz]
        direction_vector = vector([fx0, fy0, fz0])
    
    else:
        raise ValueError("The function must have 2 or 3 variables.")
    
    # Parametric equations of the normal line
    t = var('t')
    normal_line = (
        x0 + direction_vector[0] * t,
        y0 + direction_vector[1] * t,
        z0 + direction_vector[2] * t
    )
    
    return normal_line

def tangent_plane(f, point):
    
    # Determine the number of variables in f
    variables = f.variables()
    n = len(variables)
    
    if n == 2:
        # Case for f(x, y): surface is z = f(x, y)
        x, y = variables
        fx = f.diff(x)
        fy = f.diff(y)
        
        # Evaluate partial derivatives at the point
        x0, y0 = point
        z0 = f.subs(x=x0, y=y0)
        fx0 = fx.subs(x=x0, y=y0)
        fy0 = fy.subs(x=x0, y=y0)
        
        # Tangent plane equation: z = z0 + fx0*(x - x0) + fy0*(y - y0)
        tangent_plane_eq = z0 + fx0 * (x - x0) + fy0 * (y - y0)
    
    elif n == 3:
        # Case for f(x, y, z): surface is f(x, y, z) = 0
        x, y, z = variables
        fx = f.diff(x)
        fy = f.diff(y)
        fz = f.diff(z)
        
        # Evaluate partial derivatives at the point
        x0, y0, z0 = point
        fx0 = fx.subs(x=x0, y=y0, z=z0)
        fy0 = fy.subs(x=x0, y=y0, z=z0)
        fz0 = fz.subs(x=x0, y=y0, z=z0)
        
        # Tangent plane equation: fx0*(x - x0) + fy0*(y - y0) + fz0*(z - z0) = 0
        tangent_plane_eq = fx0 * (x - x0) + fy0 * (y - y0) + fz0 * (z - z0) == 0
    
    else:
        raise ValueError("The function must have 2 or 3 variables.")
    
    return tangent_plane_eq

def centerOfMass(region, density, return_mass = False):
    '''   
    Calculate the center of mass for a 2D or 3D object defined by a region and a density function.
    If the object has uniform density, specify a constant value for the density function (e.g 2.5 ).
    
    also works for calculating charge densit, total heat, and other physical properties that can be defined as an integral over a region.
    
    Parameters:
    - region: A list of tuples specifying the bounds for each variable (e.g., [(x, 0, 1), (y, 0, 2)] for 2D).
    - density: The density function of the object.
    - return_mass: If True, the function will return the mass of the object along with the center of mass  
    
    Returns:
    - A tuple representing the center of mass coordinates.

    example usage: # add more cases and results from actually testing these cases
    2 dimensional case:
    >>> centerOfMass([(x, 0, 1), (y, 0, 2)], x^2+y^2 )
    3 dimensional case:
    >>> centerOfMass([(x, 0, 1), (y, 0, 2), (z, 0, 3)], x^2+y^2+z^2 )
    '''
    variables = [var[0] for var in region]

    if len(variables) == 2:  # 2D case
        x, y = variables
        mass = integral(integral(density, region[0]), region[1])
        x_cm = integral(integral(x * density, region[0]), region[1]) / mass
        y_cm = integral(integral(y * density, region[0]), region[1]) / mass
        return ((x_cm, y_cm), mass) if return_mass else (x_cm, y_cm)
    
    elif len(variables) == 3:  # 3D case
        x, y, z = variables
        mass = integral(integral(integral(density, region[0]), region[1]), region[2])
        x_cm = integral(integral(integral(x * density, region[0]), region[1]), region[2]) / mass
        y_cm = integral(integral(integral(y * density, region[0]), region[1]), region[2]) / mass
        z_cm = integral(integral(integral(z * density, region[0]), region[1]), region[2]) / mass
        return ((x_cm, y_cm, z_cm), mass) if return_mass else (x_cm, y_cm, z_cm)
    else:
        raise ValueError("The function only supports 2D or 3D objects.")

def arc_length(r, t, parameterize=False, a=None, b=None):
    """
    Calculate the arc length of a space curve or return its arc length parameterization.

    Parameters:
        r : A SageMath vector function r(t)
        t : The parameter variable (e.g., t)
        a : Start parameter value (optional)
        b : End parameter value (optional)
        parameterize : If True, returns the arc length parameterization (default False)

    Returns:
        If parameterize=False and a,b=None: Returns arc length function s(t)
        If parameterize=False and a,b given: Returns definite arc length
        If parameterize=True: Returns r(s), the arc length parameterization(
    """
    # Compute the derivative vector r'(t)
    r_prime = np.array([diff(i) for i in r])
    # Compute the speed |r'(t)|
    speed = sqrt(sum([i^2 for i in r_prime]))

    # Arc length function s(t) = ∫_{t0}^{t} |r'(u)| du, usually from t0=0
    if parameterize:
        # 1. Find s(t)
        s_of_t = integral(speed, t)
        # 2. Solve for t in terms of s: s = s(t) => t = t(s)
        try:
            s = var('s')
            t_of_s = solve(s == s_of_t, t)[0].rhs()
            # 3. Substitute t(s) into r(t)
            r_s = r.subs(t=t_of_s)
            return r_s
        except Exception as e:
            print("Could not find explicit arc length parameterization:", e)
            return None
    elif a is not None and b is not None:
        # Definite arc length from a to b
        return integral(speed, t, a, b).full_simplify()
    else:
        # Indefinite arc length function s(t)
        return integral(speed, t).full_simplify()
